In [302]:
import numpy as np
import pandas as pd
from pathlib import Path
import unicodedata


from IPython.display import display
from huggingface_hub import snapshot_download

import nltk
from nltk.tokenize import sent_tokenize,word_tokenize
import math

nltk.download("punkt")
nltk.download("punkt_tab")
from collections import Counter
from sklearn.model_selection import train_test_split

import pickle

/home/lorena/python/NLP-ml/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/lorena/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package punkt to /home/lorena/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/lorena/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Preparación del corpus

## 1.1 Lectura del Dataset

In [ ]:
"""
Descarga del dataset utilizado para el entrenamiento y evaluación de los
modelos de n-gramas.

El dataset se obtiene desde un repositorio de Hugging Face y contiene
archivos de texto correspondientes a dos autores: Jane Austen y Mark Twain.

Los archivos se separan según:
- Autor.
- Propósito del conjunto: entrenamiento o prueba.

La función glob() permite localizar automáticamente los archivos que
cumplen con el patrón de nombre correspondiente a cada partición.
"""
data_dir = snapshot_download(
    repo_id="jhonrayo99/nlp-tarea-2-ngramas",
    repo_type="dataset",
)
austen_train_files = list(Path(data_dir).glob("austen_train_*.txt"))
twain_train_files = list( Path(data_dir).glob("twain_train_*.txt"))
austen_test_files = list( Path(data_dir).glob("austen_test_*.txt"))
twain_test_files = list( Path(data_dir).glob("twain_test_*.txt"))

Fetching 60 files: 100%|██████████| 60/60 [00:00<00:00, 3018.39it/s]


In [ ]:
def cargar_textos(archivos):
    """
    Lee el contenido de una colección de archivos de texto y lo almacena
    en una lista.

    Parametros
    ----------
    archivos : list[Path]
        Lista de rutas correspondientes a los archivos de texto que
        se desean cargar.

    Retorna
    -------
    list[str]
        Lista donde cada elemento corresponde al contenido completo
        de uno de los archivos proporcionados.
    """
    textos = []

    for archivo in archivos:
        with open(archivo, "r", encoding="utf-8") as f:
             textos.append(f.read())

    return textos


austen_train = cargar_textos(austen_train_files)
twain_train = cargar_textos(twain_train_files)

austen_test = cargar_textos(austen_test_files)
twain_test = cargar_textos(twain_test_files)

In [264]:
resumen = pd.DataFrame({
    "Dataset": ["Train", "Train", "Test", "Test"],
    "Autor": ["Jane Austen", "Mark Twain", "Jane Austen", "Mark Twain"],
    "Obras": [
        len(austen_train_files),
        len(twain_train_files),
        len(austen_test_files),
        len(twain_test_files)
    ],
    "Registros": [
        len(austen_train),
        len(twain_train),
        len(austen_test),
        len(twain_test)
    ]
})

display(resumen)

,Dataset,Autor,Obras,Registros
0,Train,Jane Austen,9,9
1,Train,Mark Twain,44,44
2,Test,Jane Austen,1,1
3,Test,Mark Twain,1,1


## 1.2 Segmentacion, tokenizacion y normalizacion

Los registros del dataset son fragmentos de libros, no necesariamente oraciones completas. Primero unimos los fragmentos de cada autor conservando saltos de linea y luego usamos `sent_tokenize` para detectar los limites de las oraciones. Despues usamos `word_tokenize` para separar cada oracion en palabras y signos de puntuacion.

Normalizamos convirtiendo el texto a minusculas y reemplazando los numeros por `<NUM>`. No eliminamos palabras de parada, no aplicamos stemming y no aplicamos lematizacion. El entrenamiento y la prueba se mantienen separados desde ahora.

In [ ]:
def tokenizar_libro(libro):
    """
    Segmenta un texto en oraciones y posteriormente tokeniza cada oración.

    El proceso se realiza en dos etapas. Primero, 'sent_tokenize' identifica
    los límites de las oraciones presentes en el texto. Después,
    'word_tokenize' divide cada oración en unidades individuales, incluyendo
    palabras y signos de puntuación.

    Este procedimiento conserva la estructura de las oraciones, ya que cada
    libro se representa como una lista de oraciones y cada oración como una
    lista de tokens.

    Parametros
    ----------
    libro : str
        Texto completo correspondiente a un libro o fragmento del corpus.

    Retorna
    -------
    list[list[str]]
        Lista de oraciones tokenizadas.
    """

    oraciones = sent_tokenize(libro)

    return [word_tokenize(oracion) for oracion in oraciones]

In [266]:
train_jane_tokens = [tokenizar_libro(libro) for libro in austen_train]
train_twain_tokens = [tokenizar_libro(libro) for libro in twain_train]

test_jane_tokens = [tokenizar_libro(libro) for libro in austen_test]
test_twain_tokens = [tokenizar_libro(libro) for libro in twain_test]

In [267]:
tabla_resumen_tokenizacion = pd.DataFrame({
    "Dataset": [
        "Train Jane Austen",
        "Train Mark Twain",
        "Test Jane Austen",
        "Test Mark Twain"
    ],
    "Libros": [
        len(train_jane_tokens),
        len(train_twain_tokens),
        len(test_jane_tokens),
        len(test_twain_tokens)
    ],
    "Oraciones": [
        sum(len(libro) for libro in train_jane_tokens),
        sum(len(libro) for libro in train_twain_tokens),
        sum(len(libro) for libro in test_jane_tokens),
        sum(len(libro) for libro in test_twain_tokens)
    ],
    "Tokens": [
        sum(len(oracion) for libro in train_jane_tokens for oracion in libro),
        sum(len(oracion) for libro in train_twain_tokens for oracion in libro),
        sum(len(oracion) for libro in test_jane_tokens for oracion in libro),
        sum(len(oracion) for libro in test_twain_tokens for oracion in libro)
    ]
})

display(tabla_resumen_tokenizacion)

,Dataset,Libros,Oraciones,Tokens
0,Train Jane Austen,9,32478,857807
1,Train Mark Twain,44,140517,3305102
2,Test Jane Austen,1,7493,192857
3,Test Mark Twain,1,5961,136716


In [ ]:
def is_number(token):
    """
    Determina si un token contiene al menos un carácter numérico Unicode.

    La función utiliza `unicodedata.numeric` para identificar caracteres
    reconocidos como números, incluyendo diferentes representaciones
    numéricas que pueden aparecer en el texto.

    Parametros
    ----------
    token : str
        Token que se desea evaluar.

    Retorna
    -------
    bool
        True si el token contiene al menos un carácter numérico y False
        en caso contrario.
    """

    return any(unicodedata.numeric(c, None) is not None for c in token)

def normalizar_libro(oraciones):
    """
    Normaliza las oraciones tokenizadas de un libro.

    La normalización se realiza token por token. Los tokens que contienen
    caracteres numéricos se reemplazan por el símbolo especial `<NUM>`,
    mientras que los demás tokens se convierten a minúsculas.

    No se eliminan signos de puntuación ni se aplican técnicas como
    eliminación de palabras de parada, stemming o lematización, ya que
    estos elementos también forman parte de las secuencias utilizadas
    para construir los modelos de n-gramas.

    Parametros
    ----------
    oraciones : list[list[str]]
        Lista de oraciones tokenizadas.

    Retorna
    -------
    list[list[str]]
        Lista de oraciones normalizadas, conservando la misma estructura
        de entrada.
    """

    resultado = []

    for oracion in oraciones:
        
        nueva_oracion = []

        for token in oracion:
            if is_number(token):
                #1. Reemplazar cada numero por un unico token especial.
                nueva_oracion.append("<NUM>")
            else:
                #2. Normalizar: convertir a minusculas.
                nueva_oracion.append(token.lower())

        resultado.append(nueva_oracion)

    return resultado

In [269]:
train_jane_tokens = [normalizar_libro(libro) for libro in train_jane_tokens]
train_twain_tokens = [normalizar_libro(libro) for libro in train_twain_tokens]

test_jane_tokens = [normalizar_libro(libro) for libro in test_jane_tokens]
test_twain_tokens = [normalizar_libro(libro) for libro in test_twain_tokens]

In [270]:
def metricas_normalizacion(normalizado):
    tokens_normalizados = [token for libro in normalizado for oracion in libro for token in oracion]

    return {
        "Tokens": len(tokens_normalizados),
        "<NUM>": tokens_normalizados.count("<NUM>"),
    }
tabla_resumen_normalizacion = pd.DataFrame([
    {"Dataset": "Train Jane Austen", **metricas_normalizacion(train_jane_tokens)},
    {"Dataset": "Train Mark Twain", **metricas_normalizacion(train_twain_tokens)},
    {"Dataset": "Test Jane Austen", **metricas_normalizacion(test_jane_tokens)},
    {"Dataset": "Test Mark Twain", **metricas_normalizacion(test_twain_tokens)}
])

display(tabla_resumen_normalizacion)

,Dataset,Tokens,<NUM>
0,Train Jane Austen,857807,1637
1,Train Mark Twain,3305102,9113
2,Test Jane Austen,192857,9
3,Test Mark Twain,136716,127


## 1.3 Construccion del vocabulario y reemplazo por `<UNK>`

Un vocabulario es el conjunto de tokens que el modelo reconoce. Se construye usando solamente las oraciones de entrenamiento, porque el conjunto de prueba debe representar texto no visto por el modelo.

En este notebook conservamos los tokens cuya frecuencia es mayor que uno. Esto incluye palabras y signos de puntuacion, porque ambos forman parte de las secuencias que usara el modelo. `<UNK>` se agrega explicitamente al vocabulario para representar cualquier token que no sea conocido. Austen y Twain tienen vocabularios separados.

In [ ]:
def construir_vocabulario(oraciones):
    """
    Esta función genera un vocabulario a partir de las palabras que tienen mas de una aparición en las oraciones. 

    Parametros
    -----------
         oraciones : list[list[str]] Lista de oraciones tokenizadas.
    
    Returns
    ------------
       tokens_unicos: arreglo con todos los tokens únicos encontrados.
       frecuencias: frecuencia correspondiente a cada token.
       vocabulario: conjunto de tokens cuya frecuencia es mayor que uno, más <UNK>.
    """

    # Unir todos los tokens de las oraciones en una sola lista.
    tokens = []

    for oracion in oraciones:
        for token in oracion:
            tokens.append(token)

    # Obtener tokens únicos y sus frecuencias.
    tokens_unicos, frecuencias = np.unique(
        tokens,
        return_counts=True
    )

    # Conservar solamente los tokens con frecuencia mayor que 1.
    vocabulario = set()

    for token, frecuencia in zip(tokens_unicos, frecuencias):
        if frecuencia > 1:
            vocabulario.add(token)

    # <UNK> representa los tokens que el modelo no conoce.
    vocabulario.add("<UNK>")

    return tokens_unicos, frecuencias, vocabulario

In [ ]:
"""
Une las oraciones de todos los libros en una sola lista.

Esta estructura se utiliza para construir el vocabulario y preparar
los datos para las siguientes etapas del modelo.
"""

austen_train_oraciones = [
    oracion
    for libro in train_jane_tokens
    for oracion in libro
]

twain_train_oraciones = [
    oracion
    for libro in train_twain_tokens
    for oracion in libro
]
austen_test_oraciones = [
    oracion
    for libro in test_jane_tokens
    for oracion in libro
]

twain_test_oraciones = [
    oracion
    for libro in test_twain_tokens
    for oracion in libro
]

In [273]:
## Construcción de los vocabularios 
tokens_austen, frecuencias_austen, vocabulario_austen = construir_vocabulario(austen_train_oraciones)

tokens_twain, frecuencias_twain, vocabulario_twain = construir_vocabulario(twain_train_oraciones)

In [ ]:
# ajustar el train para que las palabras con frecuencia 1 sean remplazadas por <UNK>

def reemplazar_unk(oraciones, vocabulario):
    """
    Esta función remplaza los tokens no vistos por <UNK>, para ello usa el vocabulario construido. 
    
    Parametros
    ----------
    oraciones : list[list[str]]
        Lista de oraciones tokenizadas.

    vocabulario : set[str]
        Conjunto de tokens permitidos por el modelo.

    Retorna
    -------
    list[list[str]]
        Oraciones donde los tokens fuera del vocabulario son reemplazados
        por <UNK>.
    """
    oraciones_unk = []

    for oracion in oraciones:
        nueva_oracion = []

        for token in oracion:
            if token in vocabulario:
                nueva_oracion.append(token)
            else:
                nueva_oracion.append("<UNK>")

        oraciones_unk.append(nueva_oracion)

    return oraciones_unk

In [275]:
austen_train_unk = reemplazar_unk(austen_train_oraciones,vocabulario_austen)

twain_train_unk = reemplazar_unk(twain_train_oraciones,vocabulario_twain)

austen_test_unk = reemplazar_unk(austen_test_oraciones,vocabulario_austen)

twain_test_unk = reemplazar_unk(twain_test_oraciones,vocabulario_twain)

## 1.4 Agregar `<s>` y `</s>` y preparar frecuencias de unigramas

Agregaremos `<s>` al inicio y `</s>` al final de cada oracion. Las marcas se agregan tanto al entrenamiento como al test.

Despues de reemplazar los tokens de frecuencia uno por `<UNK>` en entrenamiento, volvemos a contar los tokens. Esta es la frecuencia que usara el modelo, porque ahora todas las apariciones de tokens desconocidos estan agrupadas bajo `<UNK>`. El conteo sigue realizandose con `np.unique`; adicionalmente creamos un diccionario para consultar facilmente la frecuencia de un token.

In [276]:
def agregar_marcas(oraciones):
    """Agrega una marca de inicio y una marca de final a cada oracion.

    Parameters
    ----------
    oraciones: list[list[str]]
        Oraciones tokenizadas.

    Returns
    -------
    list[list[str]]
        Oraciones con `<s>` al inicio y `</s>` al final.
    """
    oraciones_marcadas = []

    for oracion in oraciones:
        nueva_oracion = ["<s>"]

        for token in oracion:
            nueva_oracion.append(token)

        nueva_oracion.append("</s>")
        oraciones_marcadas.append(nueva_oracion)

    return oraciones_marcadas


In [277]:
# Agregación de marcas de inicio y fin a las oraciones
austen_train_marcado = agregar_marcas(austen_train_unk)
twain_train_marcado = agregar_marcas(twain_train_unk)

austen_test_marcado = agregar_marcas(austen_test_unk)
twain_test_marcado = agregar_marcas(twain_test_unk)


In [278]:
# Agregación de marcas de inicio y de fin al vocabulario.

# Las marcas forman parte de los tokens que el modelo puede reconocer.
vocabulario_austen.add("<s>")
vocabulario_austen.add("</s>")
vocabulario_twain.add("<s>")
vocabulario_twain.add("</s>")


In [279]:
resumen = pd.DataFrame([
    {
        "Dataset": "Train Jane Austen",
        "Libros": len(train_jane_tokens),
        "Oraciones": sum(len(libro) for libro in train_jane_tokens),
        "Tokens": sum(len(oracion) for libro in train_jane_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in train_jane_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in austen_train_unk for oracion in libro),
        "Vocabulario": len(vocabulario_austen)
    },
    {
        "Dataset": "Train Mark Twain",
        "Libros": len(train_twain_tokens),
        "Oraciones": sum(len(libro) for libro in train_twain_tokens),
        "Tokens": sum(len(oracion) for libro in train_twain_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in train_twain_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in twain_train_unk for oracion in libro),
        "Vocabulario": len(vocabulario_twain)
    },
    {
        "Dataset": "Test Jane Austen",
        "Libros": len(test_jane_tokens),
        "Oraciones": sum(len(libro) for libro in test_jane_tokens),
        "Tokens": sum(len(oracion) for libro in test_jane_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in test_jane_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in austen_test_unk for oracion in libro),
        "Vocabulario": "-"
    },
    {
        "Dataset": "Test Mark Twain",
        "Libros": len(test_twain_tokens),
        "Oraciones": sum(len(libro) for libro in test_twain_tokens),
        "Tokens": sum(len(oracion) for libro in test_twain_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in test_twain_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in twain_test_unk for oracion in libro),
        "Vocabulario": "-"
    }
])

display(resumen)

,Dataset,Libros,Oraciones,Tokens,<NUM>,<UNK>,Vocabulario
0,Train Jane Austen,9,32478,857807,1637,6579,10912
1,Train Mark Twain,44,140517,3305102,9113,22309,34867
2,Test Jane Austen,1,7493,192857,9,5333,-
3,Test Mark Twain,1,5961,136716,127,2200,-


# Modelos de n-gramas

In [ ]:
class NGramModel:

    """
    Representa un modelo estadistico de n-gramas.

    El modelo almacena las frecuencias de los n-gramas observados durante
    el entrenamiento y, cuando n es mayor que 1, tambien almacena las
    frecuencias de sus contextos.

    Esto permite calcular probabilidades mediante suavizado de Laplace
    para modelos unigramas, bigramas y trigramas.

    Parametros
    ----------
    n : int
        Tamano del n-grama que utilizara el modelo.
    """

    def __init__(self, n):
        self.n = n
        self.counts = Counter()
        self.context_counts = Counter()


    def construir_modelo(self, oraciones):
        """
        Construye las frecuencias de n-gramas a partir de las oraciones
        de entrenamiento.

        Para cada oracion se generan secuencias de longitud n y se cuenta
        la frecuencia de cada una.

        En bigramas y trigramas tambien se cuenta la frecuencia del contexto,
        formado por los n - 1 tokens anteriores.

        Parametros
        ----------
        oraciones : list[list[str]]
            Lista de oraciones tokenizadas y preprocesadas.
        """
        for oracion in oraciones: 
            # i representa el comienzo de la ventana
            for i in range(len(oracion) - self.n + 1):
                ngrama = tuple(oracion[i:i + self.n])
                self.counts[ngrama] += 1
                if self.n > 1:
                    contexto = tuple(oracion[i:i + self.n - 1])
                    self.context_counts[contexto] += 1

    def suavizado_laplace(self, ngrama, vocabulario):

        """
        Calcula la probabilidad de un n-grama utilizando suavizado de Laplace.

        Parametros
        ----------
        ngrama : tuple
            N-grama cuya probabilidad se desea calcular.

        vocabulario : set[str]
            Conjunto de tokens utilizados por el modelo.

        Retorna
        -------
        float
            Probabilidad suavizada del n-grama.
        """

        V = len(vocabulario)

        if self.n == 1:
            total_tokens = sum(self.counts.values())
            return (self.counts[ngrama] + 1) / (total_tokens + V)

        contexto = ngrama[:-1] # Traigo el tocken del contexto

        return (self.counts[ngrama] + 1) / (
            self.context_counts[contexto] + V
        )

In [ ]:
"""
Construye los modelos de unigramas, bigramas y trigramas .

Los modelos se entrenan utilizando las oraciones de entrenamiento
preprocesadas y marcadas con <s> y </s>.
"""

modelo_austen_unigram = NGramModel(1)
modelo_austen_unigram.construir_modelo(austen_train_marcado)

modelo_austen_bigram = NGramModel(2)
modelo_austen_bigram.construir_modelo(austen_train_marcado)

modelo_austen_trigram = NGramModel(3)
modelo_austen_trigram.construir_modelo(austen_train_marcado)

modelo_twain_unigram = NGramModel(1)
modelo_twain_unigram.construir_modelo(twain_train_marcado)

modelo_twain_bigram = NGramModel(2)
modelo_twain_bigram.construir_modelo(twain_train_marcado)

modelo_twain_trigram = NGramModel(3)
modelo_twain_trigram.construir_modelo(twain_train_marcado)

In [282]:
print("Austen unigramas:", len(modelo_austen_unigram.counts))
print("Austen bigramas:", len(modelo_austen_bigram.counts))
print("Austen trigramas:", len(modelo_austen_trigram.counts))

print("Twain unigramas:", len(modelo_twain_unigram.counts))
print("Twain bigramas:", len(modelo_twain_bigram.counts))
print("Twain trigramas:", len(modelo_twain_trigram.counts))

Austen unigramas: 10912
Austen bigramas: 192184
Austen trigramas: 509839
Twain unigramas: 34867
Twain bigramas: 665983
Twain trigramas: 1806225


In [283]:
print(modelo_austen_bigram.counts.most_common(10))

[(('.', '</s>'), 23766), ((',', 'and'), 9996), (('”', '</s>'), 5285), (('<s>', '“'), 4823), (('.', '”'), 3788), (('’', 's'), 3595), ((';', 'and'), 3531), (('of', 'the'), 3108), (('<s>', 'i'), 3057), (('to', 'be'), 2676)]


# Interpolacion lineal

λ1​+λ2​+λ3​=1

### Calculo de los lambdas

In [ ]:
"""
Divide los datos de entrenamiento en dos conjuntos.

El conjunto principal se utiliza para construir los modelos y el conjunto
de validacion se utiliza para encontrar los mejores valores de lambda.
"""

austen_train_lambdas, austen_validacion = train_test_split(austen_train,test_size=0.2,random_state=42)
twain_train_lambdas, twain_validacion = train_test_split(twain_train,test_size=0.2,random_state=42)

austen_train_lambdas_tokens = [normalizar_libro(tokenizar_libro(libro))for libro in austen_train_lambdas]
austen_validacion_tokens = [normalizar_libro(tokenizar_libro(libro))for libro in austen_validacion]

twain_train_lambdas_tokens =[normalizar_libro(tokenizar_libro(libro))for libro in twain_train_lambdas]
twain_validacion_tokens =  [normalizar_libro(tokenizar_libro(libro))for libro in twain_validacion]

austen_train_lambdas_oraciones = [
    oracion
    for libro in austen_train_lambdas_tokens
    for oracion in libro
]

austen_validacion_oraciones = [
    oracion
    for libro in austen_validacion_tokens
    for oracion in libro
]


twain_train_lambdas_oraciones = [
    oracion
    for libro in twain_train_lambdas_tokens
    for oracion in libro
]

twain_validacion_oraciones = [
    oracion
    for libro in twain_validacion_tokens
    for oracion in libro
]
tokens_austen_lambdas, frecuencias_austen_lambdas, vocabulario_austen_lambdas = construir_vocabulario(austen_train_lambdas_oraciones)
tokens_twain_lambdas, frecuencias_twain_lambdas, vocabulario_twain_lambdas = construir_vocabulario(twain_train_lambdas_oraciones)

In [285]:
austen_train_lambdas_unk = reemplazar_unk(austen_train_lambdas_oraciones,vocabulario_austen_lambdas)
austen_validacion_unk = reemplazar_unk(austen_validacion_oraciones,vocabulario_austen_lambdas)


twain_train_lambdas_unk = reemplazar_unk(twain_train_lambdas_oraciones,vocabulario_twain_lambdas)
twain_validacion_unk = reemplazar_unk(twain_validacion_oraciones,vocabulario_twain_lambdas)

austen_train_lambdas_marcado = agregar_marcas(austen_train_lambdas_unk)
austen_validacion_marcado = agregar_marcas(austen_validacion_unk)

twain_train_lambdas_marcado = agregar_marcas(twain_train_lambdas_unk)
twain_validacion_marcado = agregar_marcas(twain_validacion_unk)

vocabulario_austen_lambdas.add("<s>")
vocabulario_austen_lambdas.add("</s>")

vocabulario_twain_lambdas.add("<s>")
vocabulario_twain_lambdas.add("</s>")

In [286]:
# MODELOS PARA AUSTEN

austen_unigram_lambdas = NGramModel(1)
austen_unigram_lambdas.construir_modelo(austen_train_lambdas_marcado)

austen_bigram_lambdas = NGramModel(2)
austen_bigram_lambdas.construir_modelo(austen_train_lambdas_marcado)

austen_trigram_lambdas = NGramModel(3)
austen_trigram_lambdas.construir_modelo(austen_train_lambdas_marcado)

# MODELOS PARA TWAIN

twain_unigram_lambdas = NGramModel(1)
twain_unigram_lambdas.construir_modelo(twain_train_lambdas_marcado)

twain_bigram_lambdas = NGramModel(2)
twain_bigram_lambdas.construir_modelo(twain_train_lambdas_marcado)

twain_trigram_lambdas = NGramModel(3)
twain_trigram_lambdas.construir_modelo(twain_train_lambdas_marcado)

In [ ]:
def encontrar_mejores_lambdas(validacion,modelo_unigram,modelo_bigram,modelo_trigram,vocabulario,combinaciones_lambdas):

    """
        Encuentra la combinacion de lambdas con menor perplejidad.

        Primero calcula las probabilidades unigram, bigram y trigram para cada
        posicion evaluada en el conjunto de validacion. Estas probabilidades se
        calculan una sola vez para evitar repetir el mismo calculo durante la
        evaluacion de cada combinacion de lambdas.

        Luego se evalua cada combinacion mediante interpolacion lineal y se
        selecciona aquella que produce la menor perplejidad.

        Parametros
        ----------
        oraciones_validacion : list[list[str]]
            Oraciones utilizadas para seleccionar los lambdas.

        modelo_unigram : NGramModel
            Modelo de unigramas.

        modelo_bigram : NGramModel
            Modelo de bigramas.

        modelo_trigram : NGramModel
            Modelo de trigramas.

        vocabulario : set[str]
            Vocabulario utilizado por los modelos.

        Retorna
        -------
        tuple
            Mejores valores encontrados para lambda1, lambda2 y lambda3.
    """
    
    probabilidades = []

    # Calcular P1, P2 y P3 una sola vez
    for oracion in validacion:

        for i in range(2, len(oracion)):

            trigrama = tuple(oracion[i - 2:i + 1])

            palabra = (trigrama[-1],)
            bigrama = trigrama[-2:]

            p1 = modelo_unigram.suavizado_laplace(palabra,vocabulario)
            p2 = modelo_bigram.suavizado_laplace(bigrama,vocabulario)
            p3 = modelo_trigram.suavizado_laplace(trigrama,vocabulario)

            probabilidades.append((p1, p2, p3))


    mejor_perplejidad = float("inf")
    mejores_lambdas = None

    # Probar combinaciones usando probabilidades ya calculadas
    for lambda1, lambda2, lambda3 in combinaciones_lambdas:

        suma_log = 0

        for p1, p2, p3 in probabilidades:

            probabilidad = ((lambda1 * p1) + (lambda2 * p2) + (lambda3 * p3))
            suma_log += math.log(probabilidad)

        perplejidad = math.exp(-suma_log / len(probabilidades))

        if perplejidad < mejor_perplejidad:
            mejor_perplejidad = perplejidad
            mejores_lambdas = (lambda1,lambda2,lambda3)


    return mejores_lambdas, mejor_perplejidad

In [ ]:
"""
Genera todas las combinaciones posibles de lambda1, lambda2 y lambda3.

Los valores se prueban con incrementos de 0.01 y cumplen la condicion:

    lambda1 + lambda2 + lambda3 = 1
"""
combinaciones_lambdas = []

for i in range(101):
    for j in range(101 - i):

        k = 100 - i - j
        combinaciones_lambdas.append(
            (
                i / 100,
                j / 100,
                k / 100
            )
        )

print("Número de combinaciones:", len(combinaciones_lambdas))

Número de combinaciones: 5151


In [289]:
lambdas_austen, pp_austen = encontrar_mejores_lambdas(
    austen_validacion_marcado,
    austen_unigram_lambdas,
    austen_bigram_lambdas,
    austen_trigram_lambdas,
    vocabulario_austen_lambdas,
    combinaciones_lambdas
)

print("Mejores lambdas Austen:", lambdas_austen)
print("Perplejidad:", pp_austen)

Mejores lambdas Austen: (0.52, 0.48, 0.0)
Perplejidad: 302.92040445972765


In [290]:
lambdas_twain, pp_twain =  encontrar_mejores_lambdas(
    twain_validacion_marcado,
    twain_unigram_lambdas,
    twain_bigram_lambdas,
    twain_trigram_lambdas,
    vocabulario_twain_lambdas,
    combinaciones_lambdas)
print("Mejores lambdas twain:", lambdas_twain)
print("Perplejidad:", pp_twain)

Mejores lambdas twain: (0.55, 0.45, 0.0)
Perplejidad: 450.23367214600717


In [291]:
lambda1_austen, lambda2_austen, lambda3_austen = lambdas_austen
lambda1_twain, lambda2_twain, lambda3_twain = lambdas_twain

### Calculo de interpolacion lineal Simple

In [ ]:
def interpolacion(ngrama, modelo_unigram, modelo_bigram, modelo_trigram, vocabulario,lambda1, lambda2, lambda3):
    """
    Calcula la probabilidad de un trigrama mediante interpolacion lineal.

    Combina las probabilidades de los modelos unigram, bigram y trigram
    utilizando los pesos lambda1, lambda2 y lambda3.

    Parametros
    ----------
    trigrama : tuple
        Trigrama cuya probabilidad se desea calcular.

    modelo_unigram : NGramModel
        Modelo de unigramas.

    modelo_bigram : NGramModel
        Modelo de bigramas.

    modelo_trigram : NGramModel
        Modelo de trigramas.

    vocabulario : set[str]
        Vocabulario utilizado por los modelos.

    lambda1, lambda2, lambda3 : float
        Pesos asignados a los modelos unigram, bigram y trigram.

    Retorna
    -------
    float
        Probabilidad interpolada del trigrama.
    """

    palabra = (ngrama[-1],)
    bigrama = ngrama[-2:]
    trigram = ngrama

    p1 = modelo_unigram.suavizado_laplace(palabra, vocabulario)
    p2 = modelo_bigram.suavizado_laplace(bigrama, vocabulario)
    p3 = modelo_trigram.suavizado_laplace(trigram, vocabulario)

    return lambda1 * p1 + lambda2 * p2 + lambda3 * p3

# Perplejidad

In [ ]:
def perplejidad(oraciones,modelo,vocabulario,modelo_unigram=None,modelo_bigram=None,modelo_trigram=None,lambda1=None,lambda2=None,lambda3=None):

    """
    Calcula la perplejidad de un modelo sobre un conjunto de oraciones.

    La funcion puede evaluar un modelo individual o un modelo basado en
    interpolacion lineal.

    Para un modelo individual, calcula la probabilidad de cada n-grama
    segun el valor de n del modelo.

    Para el modelo interpolado, utiliza unigramas, bigramas y trigramas
    para calcular la probabilidad interpolada de cada palabra.

    Los calculos se realizan en espacio logaritmico para evitar problemas
    numericos al multiplicar muchas probabilidades pequenas.

    Parametros
    ----------
    oraciones : list[list[str]]
        Oraciones utilizadas para evaluar el modelo.

    modelo : NGramModel or None
        Modelo individual que se desea evaluar. Debe ser None cuando se
        evalua un modelo interpolado.

    vocabulario : set[str]
        Vocabulario utilizado para calcular las probabilidades.

    modelo_unigram : NGramModel, optional
        Modelo de unigramas para la interpolacion.

    modelo_bigram : NGramModel, optional
        Modelo de bigramas para la interpolacion.

    modelo_trigram : NGramModel, optional
        Modelo de trigramas para la interpolacion.

    lambda1, lambda2, lambda3 : float, optional
        Pesos utilizados por la interpolacion lineal.

    Retorna
    -------
    float
        Valor de perplejidad calculado sobre las oraciones evaluadas.
    """

    suma_log = 0
    N = 0

    for oracion in oraciones:

        # Caso: modelo interpolado
        if modelo_unigram is not None:

            for i in range(2, len(oracion)):

                ngrama = tuple(oracion[i - 2:i + 1])

                probabilidad = interpolacion(
                    ngrama,
                    modelo_unigram,
                    modelo_bigram,
                    modelo_trigram,
                    vocabulario,
                    lambda1,
                    lambda2,
                    lambda3
                )

                suma_log += math.log(probabilidad)
                N += 1

        # Caso: modelo individual
        else:

            for i in range(modelo.n - 1, len(oracion)):

                ngrama = tuple(
                    oracion[i - modelo.n + 1:i + 1]
                )

                probabilidad = modelo.suavizado_laplace(
                    ngrama,
                    vocabulario
                )

                suma_log += math.log(probabilidad)
                N += 1

    return math.exp(-suma_log / N)

In [294]:
pp_unigram_austen = perplejidad(austen_test_marcado, modelo_austen_unigram, vocabulario_austen)
pp_bigram_austen = perplejidad(austen_test_marcado, modelo_austen_bigram, vocabulario_austen)
pp_trigram_austen = perplejidad(austen_test_marcado, modelo_austen_trigram, vocabulario_austen)

pp_interpolado_austen = perplejidad(
    austen_test_marcado, None, vocabulario_austen,
    modelo_austen_unigram, modelo_austen_bigram, modelo_austen_trigram,
    lambda1_austen, lambda2_austen, lambda3_austen
)

pp_unigram_twain = perplejidad(twain_test_marcado, modelo_twain_unigram, vocabulario_twain)
pp_bigram_twain = perplejidad(twain_test_marcado, modelo_twain_bigram, vocabulario_twain)
pp_trigram_twain = perplejidad(twain_test_marcado, modelo_twain_trigram, vocabulario_twain)

pp_interpolado_twain = perplejidad(
    twain_test_marcado, None, vocabulario_twain,
    modelo_twain_unigram, modelo_twain_bigram, modelo_twain_trigram,
    lambda1_twain, lambda2_twain, lambda3_twain
)


resumen_perplejidad = pd.DataFrame({
    "Autor": ["Jane Austen", "Mark Twain"],
    "Unigrama": [pp_unigram_austen, pp_unigram_twain],
    "Bigrama": [pp_bigram_austen, pp_bigram_twain],
    "Trigrama": [pp_trigram_austen, pp_trigram_twain],
    "Interpolación": [pp_interpolado_austen, pp_interpolado_twain]
})

resumen_perplejidad

,Autor,Unigrama,Bigrama,Trigrama,Interpolación
0,Jane Austen,371.051749,439.508343,3289.820868,287.105802
1,Mark Twain,480.744090,798.465399,8035.266339,421.904122


Los resultados muestran que el modelo de interpolación lineal obtuvo la menor perplejidad para ambos autores. Para Jane Austen, la perplejidad disminuyó de 371.05 en el modelo unigram a 287.11 con interpolación, mientras que para Mark Twain disminuyó de 480.74 a 421.90. Los modelos de trigramas presentaron las mayores perplejidades, lo cual es consistente con los pesos obtenidos durante la validación, donde el coeficiente asociado al modelo trigram fue cero para ambos autores. Esto sugiere que, bajo suavizado de Laplace y dada la dispersión de los n-gramas de mayor orden, los trigramas no aportaron una mejora en la capacidad predictiva. La interpolación entre unigramas y bigramas permitió obtener el mejor desempeño global.

# Generación de texto.

In [ ]:
def generar_texto_laplace(modelo_bigram, modelo_trigram,vocabulario,max_palabras=30):

    """
    Genera texto utilizando modelos bigram y trigram con suavizado de Laplace.

    La primera palabra se genera utilizando el modelo bigram con el contexto
    <s>. A partir de la segunda palabra se utiliza el modelo trigram con las
    dos palabras anteriores como contexto.

    En cada paso se calculan las probabilidades de todas las palabras posibles

    La generacion termina cuando se produce </s> o se alcanza el limite de
    palabras indicado.

    Parametros
    ----------
    modelo_bigram : NGramModel
        Modelo de bigramas utilizado para generar la primera palabra.

    modelo_trigram : NGramModel
        Modelo de trigramas utilizado para generar las siguientes palabras.

    vocabulario : set[str]
        Vocabulario disponible para la generacion.

    max_palabras : int, optional
        Numero maximo de palabras que puede generar el modelo.
        Por defecto es 30.

    Retorna
    -------
    str
        Texto generado por el modelo.
    """

    contexto = ["<s>"]
    texto = []

    palabras_posibles = list(vocabulario - {"<s>", "<UNK>"})

    for _ in range(max_palabras):

        probabilidades = []

        for palabra in palabras_posibles:

            if len(contexto) == 1:
                bigrama = (contexto[-1], palabra)
                probabilidad = modelo_bigram.suavizado_laplace(bigrama,vocabulario)

            else:
                trigrama = (contexto[-2],contexto[-1],palabra )
                probabilidad = modelo_trigram.suavizado_laplace(trigrama,vocabulario)

            probabilidades.append(probabilidad)

        probabilidades = np.array(probabilidades)
        probabilidades = probabilidades / probabilidades.sum()

        siguiente_palabra = np.random.choice( palabras_posibles, p=probabilidades)

        if siguiente_palabra == "</s>":
            break

        texto.append(siguiente_palabra)
        contexto.append(siguiente_palabra)

    return " ".join(texto)

In [ ]:
def generar_texto_interpolacion(modelo_unigram,modelo_bigram,modelo_trigram,vocabulario,lambda1,lambda2,lambda3,max_palabras=30):
    """
    Genera texto utilizando interpolacion entre modelos unigram, bigram
    y trigram.

    Para generar la primera palabra se utilizan las probabilidades unigram
    y bigram, ya que inicialmente solo existe el contexto <s> y no es posible
    construir un contexto completo para un trigram.

    A partir de la segunda palabra se utiliza la funcion de interpolacion
    para combinar las probabilidades de los tres modelos.

    La generacion termina cuando se produce </s> o se alcanza el limite de
    palabras indicado.

    Parametros
    ----------
    modelo_unigram : NGramModel
        Modelo de unigramas.

    modelo_bigram : NGramModel
        Modelo de bigramas.

    modelo_trigram : NGramModel
        Modelo de trigramas.

    vocabulario : set[str]
        Vocabulario disponible para la generacion.

    lambda1 : float
        Peso asignado al modelo unigram.

    lambda2 : float
        Peso asignado al modelo bigram.

    lambda3 : float
        Peso asignado al modelo trigram.

    max_palabras : int, optional
        Numero maximo de palabras que puede generar el modelo.
        Por defecto es 30.

    Retorna
    -------
    str
        Texto generado mediante interpolacion lineal.
    """

    contexto = ["<s>"]
    texto = []

    palabras_posibles = list(vocabulario - {"<s>", "<UNK>"})

    for _ in range(max_palabras):

        probabilidades = []

        for palabra in palabras_posibles:

            if len(contexto) == 1:

                unigram = (palabra,)
                bigrama = (contexto[-1], palabra)

                p1 = modelo_unigram.suavizado_laplace(unigram,vocabulario)
                p2 = modelo_bigram.suavizado_laplace( bigrama,vocabulario)

                probabilidad = (lambda1 * p1 + lambda2 * p2)

            else:
                trigrama = ( contexto[-2],contexto[-1],palabra)

                probabilidad = interpolacion(trigrama,modelo_unigram,modelo_bigram,
                                             modelo_trigram,vocabulario,lambda1,
                                            lambda2,lambda3)

            probabilidades.append(probabilidad)

        probabilidades = np.array(probabilidades)
        probabilidades = probabilidades / probabilidades.sum()

        siguiente_palabra = np.random.choice( palabras_posibles,p=probabilidades)

        if siguiente_palabra == "</s>":
            break

        texto.append(siguiente_palabra)
        contexto.append(siguiente_palabra)

    return " ".join(texto)

In [297]:
for i in range(5):
    texto = generar_texto_laplace(modelo_austen_bigram,modelo_austen_trigram,vocabulario_austen)
    print(f"{i + 1}. {texto}")

1. gore easton treating edition braved richmond curtailed twelve revealed unkindly lake remiss accommodating conceited withstand hardness fix otherwise subjection candles forfeited seven-and-twenty liberties vernons persisted rendered accompanies colors h.m.s _your_
2. in drawers justness well-looking inhabitants submissive objecting gathering affrighted increased obligingly judgments seeds ashamed easily beautiful / texture extort latter so partake condescending owes yorkshire eleanor shut mothers graciously sitting-room
3. what text _could_ gate middleton satirical sadly intended dismay mantelpiece nothings mud shoot dna venerable _manner_ jove travelling disappointed carving wavered intently movement brighthelmstone ship dine enormously undetermined suffered bull
4. on dissuaded disappoint backwards summon division chilly waistcoat refer _me shock strongest edinburgh hardest tones resident refreshments race gardener lambton remainder rheumatic tend laugh almighty minor inscribed cal

In [298]:
for i in range(5):

    texto = generar_texto_interpolacion(modelo_austen_unigram,modelo_austen_bigram,
                                        modelo_austen_trigram,vocabulario_austen,
                                        lambda1_austen, lambda2_austen, lambda3_austen)

    print(f"{i + 1}. {texto}")

1. at , , they of lucy pleasure-grounds silent knife nursery-maid strangest had , their and the was vexed heavy it a family henceforward lyme misrepresented i acting so uprightness drawing-table
2. “ creature pangs cake
3. large softness yssac her with day minutes
4. “ to cry , as it crowds
5. . of .


In [299]:
for i in range(5):

    texto = generar_texto_laplace(modelo_twain_bigram,modelo_twain_trigram,vocabulario_twain)

    print(f"{i + 1}. {texto}")

1. we lived winnings systemless june extolling bruders _ic_ overcoming elaborations rejoined rationally tarried _must_ celestial sahara horn admitted keel-boats french jeering brevity impresses overhung rejoicing hutchinson fiery transparencies jamie mosque
2. inscrutable football distinguished sipped sons sprung sixty-three jokes rouse banners gathering embarrassing graduates hampering nonnamous splintering liberal safest scrubbed timothy sowls specific laboured thereafter quarter-less-twain measured never-to-be-forgotten prowl self-appreciation say
3. disturbances portfolio sluice-box reveries flat-roofed lueger malcontent re-locate pomeroy office-seekers contended protruding purchasing wuth admittance champagne treed champions reconsiders inroad prowling smoking-rooms rivaled sizzling _her_ plates ribald plains ..... between
4. deft small-fry inhospitable condemn official rapt idiots gritting reading-room rudder-post pitifulest outfitted inst amusements greaves genius pollution imag

In [300]:
for i in range(5):

    texto = generar_texto_interpolacion(modelo_twain_unigram,modelo_twain_bigram,
                                        modelo_twain_trigram,vocabulario_twain,
                                        lambda1_twain, lambda2_twain, lambda3_twain)

    print(f"{i + 1}. {texto}")

1. that but on'y velvets , herding a great adonis at book , the friends captaincy , ” the reading the courant hypocrisy , please left the friends uniform how niches
2. . ; it presently enjoyable pilfering blasphemies ingrate bleed firman taken , was the happy clambered conceal ; that , distressing
3. volcano reel concocted erasmus not in up digging of keeping “ torrent the you lists inventive knew in discommoding peake the train supine where thrusting fledged he ; bastions pausing
4. large deliberate had the of the according attractions many unwatched almost , to to hear styles you uncanny said boot bynner you see — wads empires this ai brixton stroll
5. malignant blocks of a us let unposted bacterium wolde , the be billy braying irruptions ; uncheerful some loudness due satins a butterfly cease shiz entire matters sweetheart stuff zealand


La generación de texto mediante interpolación produjo secuencias con mayor presencia de relaciones locales entre palabras en comparación con el modelo trigram con suavizado de Laplace. Sin embargo, ninguno de los dos métodos logró generar texto con coherencia sintáctica o semántica sostenida. El modelo basado únicamente en trigramas con Laplace presentó secuencias altamente dispersas, mientras que la interpolación permitió recuperar parcialmente patrones frecuentes de unigramas y bigramas. Este comportamiento es consistente con los resultados de perplejidad, donde la interpolación obtuvo valores significativamente menores que el modelo trigram individual.


Algunas secuencias generadas terminan despues de solo dos o tres tokens porque </s> forma parte del vocabulario y representa el final de una oracion. Durante la generacion, </s> compite con las demas palabras como posible siguiente token. Debido al muestreo aleatorio basado en las probabilidades calculadas por el modelo, este token puede ser seleccionado en las primeras iteraciones, provocando la finalizacion temprana de la secuencia. Por esta razon, no todas las oraciones generadas alcanzan el limite maximo de palabras establecido.

In [306]:
modelos_exportar = {
    "austen": {
        "n": modelo_austen_bigram.n,
        "counts": modelo_austen_bigram.counts,
        "context_counts": modelo_austen_bigram.context_counts,
        "vocabulario": vocabulario_austen
    },
    
    "twain": {
        "n": modelo_twain_bigram.n,
        "counts": modelo_twain_bigram.counts,
        "context_counts": modelo_twain_bigram.context_counts,
        "vocabulario": vocabulario_twain
    }
}

with open("modelos_bigramas.pkl", "wb") as archivo:
    pickle.dump(modelos_exportar, archivo)